# A/B Test Analysis: Subscription Page Conversion Optimisation
**Analyst:** Zari Syed | **Date:** May 2025 | **Domain:** Digital media subscription analytics

---

## Overview

This notebook tests whether a redesigned subscription landing page improves conversion from free reader to paid subscriber — the core commercial challenge for any subscription-based news publisher.

**Test setup:**
- **Control:** Existing subscription page (baseline conversion ~2.8%)
- **Variant:** Redesigned page with social proof elements (subscriber count, testimonials, highlighted benefits)
- **Primary metric:** Conversion rate (visitor to paid subscriber)
- **Traffic split:** 50/50 random assignment | **Test duration:** 2 weeks

**Hypotheses:**
- H0: p_variant = p_control (no effect)
- H1: p_variant != p_control (two-tailed, alpha = 0.05)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import math, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11
})
sns.set_palette('muted')
np.random.seed(2024)
print("Libraries loaded.")

## 1. Data Generation & Exploration

We simulate 2 weeks of realistic subscription page traffic.
A baseline conversion rate of ~2.8% is typical for a quality digital news product.

In [ ]:
N_CONTROL = 8000
N_VARIANT = 8000
P_CONTROL = 0.028
P_VARIANT = 0.036

control_outcomes = np.random.binomial(1, P_CONTROL, N_CONTROL)
variant_outcomes = np.random.binomial(1, P_VARIANT, N_VARIANT)

df = pd.DataFrame({
    'variant': ['Control'] * N_CONTROL + ['Variant'] * N_VARIANT,
    'converted': list(control_outcomes) + list(variant_outcomes)
})

summary = df.groupby('variant')['converted'].agg(
    Visitors='count', Conversions='sum'
).assign(Conversion_Rate=lambda x: (x['Conversions'] / x['Visitors']).round(4))
summary['Conversion_Rate_pct'] = (summary['Conversion_Rate'] * 100).round(2).astype(str) + '%'
print("Experiment summary:")
print(summary.to_string())
print("Observed lift: {:+.2f}pp".format((variant_outcomes.mean() - control_outcomes.mean()) * 100))

## 2. Two-Proportion Z-Test

Standard test for comparing binary conversion rates between two independent groups.
We compute the p-value using `math.erfc` (standard library) — no scipy needed.

In [ ]:
c_conv = int(control_outcomes.sum())
v_conv = int(variant_outcomes.sum())
c_n, v_n = N_CONTROL, N_VARIANT

p_c = c_conv / c_n
p_v = v_conv / v_n
p_pool = (c_conv + v_conv) / (c_n + v_n)

se = math.sqrt(p_pool * (1 - p_pool) * (1/c_n + 1/v_n))
z_stat = (p_v - p_c) / se
p_value = math.erfc(abs(z_stat) / math.sqrt(2))
ALPHA = 0.05

print("=" * 55)
print("TWO-PROPORTION Z-TEST")
print("=" * 55)
print("  Control conversion rate : {:.2f}%  ({}/{})".format(p_c*100, c_conv, c_n))
print("  Variant conversion rate : {:.2f}%  ({}/{})".format(p_v*100, v_conv, v_n))
print("  Absolute lift           : {:+.2f} percentage points".format((p_v-p_c)*100))
print("  Relative lift           : {:+.1f}%".format((p_v-p_c)/p_c*100))
print("  Z-statistic             : {:.4f}".format(z_stat))
print("  P-value (two-tailed)    : {:.4f}".format(p_value))
print("  Significance (alpha=0.05): {}".format("YES -- reject H0" if p_value < ALPHA else "NO -- fail to reject H0"))

## 3. Chi-Square Test (Cross-validation)

We validate using a chi-square test of independence on the 2x2 contingency table.
For a 2x2 table with df=1: chi-squared = z-squared, so both tests must agree.

In [ ]:
obs = np.array([[c_conv, c_n - c_conv], [v_conv, v_n - v_conv]])
row_totals = obs.sum(axis=1, keepdims=True)
col_totals = obs.sum(axis=0, keepdims=True)
expected = (row_totals * col_totals) / obs.sum()
chi2 = float(((obs - expected) ** 2 / expected).sum())
chi2_p = math.erfc(math.sqrt(chi2 / 2))

ct = pd.DataFrame(obs, index=['Control','Variant'], columns=['Converted','Not Converted'])
ct['Total'] = ct.sum(axis=1)
print("Contingency Table:")
print(ct.to_string())
print("")
print("Chi-square statistic : {:.4f}".format(chi2))
print("P-value              : {:.4f}".format(chi2_p))
print("Cross-check: z^2={:.4f} == chi2={:.4f}  OK".format(z_stat**2, chi2))

## 4. Confidence Intervals

A 95% CI for the difference in conversion rates shows the plausible range of the true lift.
If the entire interval lies above zero, we have strong evidence the effect is positive.

In [ ]:
z_crit = 1.96
se_diff = math.sqrt(p_c*(1-p_c)/c_n + p_v*(1-p_v)/v_n)
diff = p_v - p_c
ci_lo = diff - z_crit * se_diff
ci_hi = diff + z_crit * se_diff
err_c = z_crit * math.sqrt(p_c*(1-p_c)/c_n)
err_v = z_crit * math.sqrt(p_v*(1-p_v)/v_n)

print("Control 95% CI : [{:.2f}%, {:.2f}%]".format((p_c-err_c)*100, (p_c+err_c)*100))
print("Variant 95% CI : [{:.2f}%, {:.2f}%]".format((p_v-err_v)*100, (p_v+err_v)*100))
print("Lift 95% CI    : [{:+.2f}pp, {:+.2f}pp]".format(ci_lo*100, ci_hi*100))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colours = ['#5B9BD5', '#ED7D31']
ax = axes[0]
bars = ax.bar(['Control', 'Variant'], [p_c*100, p_v*100], color=colours, width=0.45, alpha=0.85)
ax.errorbar(['Control','Variant'], [p_c*100, p_v*100], yerr=[err_c*100, err_v*100],
            fmt='none', color='black', capsize=7, linewidth=1.8)
for bar, rate in zip(bars, [p_c*100, p_v*100]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            "{:.2f}%".format(rate), ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Variant (Error bars = 95% CI)')
ax.set_ylim(0, max(p_c, p_v)*100 * 1.5)

ax2 = axes[1]
ax2.barh(['Lift'], [diff*100], xerr=[[(diff-ci_lo)*100], [(ci_hi-diff)*100]],
         color='#70AD47', alpha=0.85, capsize=8, height=0.3)
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='No effect (0)')
ax2.set_xlabel('Lift (percentage points)')
ax2.set_title('95% Confidence Interval for Conversion Rate Lift')
ax2.legend()
plt.suptitle('A/B Test Results: Subscription Page Redesign', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Power Analysis & Sample Size Planning

Before running an experiment, calculating required sample size prevents under-powered tests
that waste traffic and produce inconclusive results.

**Target:** 80% power at alpha = 0.05 (i.e. 20% chance of missing a real effect).

In [ ]:
def required_n(p_base, mde, alpha=0.05, power=0.80):
    # Required sample per variant for a two-proportion z-test
    p2 = p_base + mde
    p_avg = (p_base + p2) / 2
    z_a = 1.96
    z_b = 0.842
    n = ((z_a * math.sqrt(2 * p_avg * (1-p_avg)) +
          z_b * math.sqrt(p_base*(1-p_base) + p2*(1-p2))) / mde) ** 2
    return math.ceil(n)

mdes = [0.002, 0.004, 0.006, 0.008, 0.010, 0.012, 0.015]
ns   = [required_n(P_CONTROL, e) for e in mdes]

print("{:>9} {:>14} {:>12} {:>10}".format("MDE (pp)", "Relative MDE", "n/variant", "Total n"))
print("-" * 50)
for e, n in zip(mdes, ns):
    print("{:>8.1f}%  {:>12.0f}%  {:>12,}  {:>10,}".format(e*100, e/P_CONTROL*100, n, n*2))

our_n = required_n(P_CONTROL, P_VARIANT - P_CONTROL)
print("\nFor our MDE ({:.1f}pp / {:.0f}% relative):".format(
    (P_VARIANT-P_CONTROL)*100, (P_VARIANT-P_CONTROL)/P_CONTROL*100))
print("  Required per variant : {:,}".format(our_n))
print("  Actual per variant   : {:,}".format(N_CONTROL))
print("  Test was             : {}".format("Adequately powered" if N_CONTROL >= our_n else "Under-powered"))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([e*100 for e in mdes], ns, 'o-', color='#2C5F8A', lw=2, ms=7)
ax.axhline(N_CONTROL, color='#ED7D31', ls='--', lw=1.5,
           label='Actual n per variant ({:,})'.format(N_CONTROL))
ax.axvline((P_VARIANT-P_CONTROL)*100, color='grey', ls=':', alpha=0.7,
           label='Our MDE ({:.1f}pp)'.format((P_VARIANT-P_CONTROL)*100))
ax.set_xlabel('Minimum Detectable Effect (percentage points)')
ax.set_ylabel('Required sample size per variant')
ax.set_title('Sample Size vs Effect Size (alpha=0.05, 80% power)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:,.0f}'.format(x)))
plt.tight_layout()
plt.show()

## 6. Business Impact & Recommendation

Translating the statistical result into a commercial recommendation, including estimated annual revenue impact.

In [ ]:
MONTHLY_PAGE_VISITORS = 500_000
MONTHLY_PRICE = 9.99
RETENTION_RATE = 0.75

extra_subs_mo = MONTHLY_PAGE_VISITORS * (p_v - p_c)
annual_revenue = extra_subs_mo * MONTHLY_PRICE * 12 * RETENTION_RATE

print("=" * 60)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 60)
print("  Visitors tested      : {:,}".format(N_CONTROL+N_VARIANT))
print("  Control rate         : {:.2f}%".format(p_c*100))
print("  Variant rate         : {:.2f}%".format(p_v*100))
print("  Absolute lift        : {:+.2f}pp".format((p_v-p_c)*100))
print("  Relative lift        : {:+.1f}%".format((p_v-p_c)/p_c*100))
print("  95% CI               : [{:+.2f}pp, {:+.2f}pp]".format(ci_lo*100, ci_hi*100))
print("  P-value              : {:.4f}".format(p_value))
print("  Statistically sig.   : {}".format("Yes" if p_value < 0.05 else "No"))
print("")
print("RECOMMENDATION: Ship the variant.")
print("")
print("  Estimated annual revenue impact:")
print("  +{:,.0f} new subscribers/month x GBP{:.2f}/month x 12 x {:.0f}% retention".format(
    extra_subs_mo, MONTHLY_PRICE, RETENTION_RATE*100))
print("  = ~GBP{:,.0f}/year".format(annual_revenue))
print("")
print("NEXT STEPS:")
print("  1. Monitor post-launch metrics for novelty effects (2-4 weeks)")
print("  2. Segment results by traffic source and device type")
print("  3. Test further variants on the checkout flow")